
## W6T5 GENERIC TRAINING ENGINE



## 1. Purpose

This notebook serves as a demonstration of the workflow for `trainer.py`.

The engine provides one reusable interface for training models with either:

- **scikit-learn** models (Random Forest, Isolation Forest, MLP)
- **PyTorch** models (Multi Layer Perceptron) 

Main parts of the file:

- `TrainingConfig`: stores model and training settings
- `load_sklearn_model()`: loads supported sklearn models
- `SimpleMLP`: a simple feedforward neural network for PyTorch
- `load_pytorch_model()`: loads supported PyTorch models
- `GenericTrainingEngine`: handles training and prediction



## 2. Workflow Process

The general flow is:

1. Create a `TrainingConfig`
2. Create the `GenericTrainingEngine`
3. Call `.fit()` to train
4. Call `.predict()` to make predictions

The engine decides internally whether to use the sklearn or PyTorch workflow based on `model_type`.



## 3. Example: scikit-learn workflow

This path works even if PyTorch is not installed, because the code only uses the sklearn branch when `model_type='sklearn'`.



In [ ]:
import sys
sys.path.append(r"C:\Users\Danny\Desktop\Phoenix\ai-ml")

from training_pipeline.src.core.trainer import TrainingConfig, GenericTrainingEngine

In [6]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data = load_iris() # Example data just to show workflow process
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)


In [7]:
# Configure sklearn model
config = TrainingConfig(
    model_type='sklearn',
    model_name='random_forest',
    model_params={'n_estimators': 100, 'random_state': 123},
    verbose=True
)

engine = GenericTrainingEngine(config)
train_result = engine.fit(X_train, y_train)
preds = engine.predict(X_test)

print(train_result)
print(preds[:10])


Finished training sklearn model: random_forest
{'model_type': 'sklearn', 'status': 'trained'}
[1 2 2 1 0 1 1 0 0 1]


### What happens in the sklearn path

- `__init__()` calls `_load_model()`
- `_load_model()` checks `model_type='sklearn'`
- `load_sklearn_model()` loads the selected model from the registry
- `.fit()` sends training to `_fit_sklearn()`
- `_fit_sklearn()` runs `self.model.fit(X_train, y_train)`
- `.predict()` uses `self.model.predict(X)`

The output confirms that the sklearn pipeline was used, specifically the Random Forest model. The returned status indicates that training was completed successfully, and the prediction array represents the model’s predicted class labels for the test dataset.

## 4. Example: PyTorch workflow

This branch is only used if `model_type='pytorch'` and PyTorch is installed.

If PyTorch is not installed, the file still imports safely, but the PyTorch training path will raise an error only when you try to use it.


In [10]:
try:
    import torch

    pytorch_config = TrainingConfig(
        model_type='pytorch',
        model_name='simple_mlp',
        model_params={'input_dim': X_train.shape[1], 'hidden_dim': 64, 'output_dim': 3},
        epochs=5,
        batch_size=16,
        learning_rate=0.001,
        verbose=True
    )

    pytorch_engine = GenericTrainingEngine(pytorch_config)
    train_result = pytorch_engine.fit(X_train, y_train)
    preds = pytorch_engine.predict(X_test)

    print(train_result)
    print(preds[:10])

except ImportError as e:
    print('PyTorch workflow not available:', e)


Epoch 1: loss=8.0637
Epoch 2: loss=6.8930
Epoch 3: loss=6.1841
Epoch 4: loss=5.2660
Epoch 5: loss=4.6659
{'model_type': 'pytorch', 'status': 'trained'}
[2 2 2 1 0 2 2 0 0 2]


### What happens in the PyTorch path

- `__init__()` calls `_load_model()`
- `_load_model()` checks `model_type='pytorch'`
- `load_pytorch_model()` loads the selected model from the registry (e.g. `SimpleMLP`)
- `.fit()` sends training to `_fit_pytorch()`
- `_fit_pytorch()`:
  - sets device (`cpu` or `cuda`) and moves model
  - initializes optimizer and loss function
  - converts input data to PyTorch tensors
  - creates a `DataLoader` for batching
  - runs training loop over epochs:
    - `self.model(batch_X)` performs the forward pass
    - `loss_fn(outputs, batch_y)` computes loss
    - `loss.backward()` computes gradients
    - `optimizer.step()` updates weights
    - `optimizer.zero_grad()` resets gradients
- `.predict()` sends data to `_predict_pytorch()`
- `_predict_pytorch()`:
  - runs forward pass with `self.model(X)`
  - uses `torch.argmax()` to get predicted class labels


Overall, the PyTorch path performs manual training using forward pass, loss calculation, backpropagation, and weight updates across epochs.

## 5. How the forward pass works

The forward pass is just the data moving through those layers in order.

In the `SimpleMLP` model, the `forward()` function is:

```python
def forward(self, x):
    return self.net(x)
```

This means the input `x` is passed through the neural network stored in `self.net`.

That network is:

1. Linear layer: input → hidden
2. ReLU activation
3. Linear layer: hidden → hidden
4. ReLU activation
5. Linear layer: hidden → output


## 6. How the PyTorch training loop works

Inside `_fit_pytorch()`:

1. Data is converted into tensors
2. A `TensorDataset` and `DataLoader` are created
3. For each epoch:
   - model is put into training mode with `self.model.train()`
   - batches are moved to the selected device
   - `optimizer.zero_grad()` clears old gradients
   - `outputs = self.model(batch_X)` runs the forward pass
   - `loss = loss_fn(outputs, batch_y)` calculates loss
   - `loss.backward()` computes gradients
   - `optimizer.step()` updates the weights

So the key forward pass line is:

```python
outputs = self.model(batch_X)
```

That calls `forward()` automatically.


## 7. Why the system still works without PyTorch

At the top of the file, PyTorch is imported inside a `try/except` block.

If PyTorch is missing:

- `torch = None`
- `nn = None`
- `DataLoader = None`
- `TensorDataset = None`

That means the file can still load successfully.

The sklearn workflow still works normally.

Only the PyTorch-specific path fails if you actually try to use it.


## 8. Example error cases

These examples show the kinds of errors the engine can raise, and why they happen.


### Error 1: Unsupported `model_type`

This happens when `model_type` is not `'sklearn'` or `'pytorch'`.


In [17]:
try:
    bad_config = TrainingConfig(
        model_type='tensorflow',
        model_name='anything'
    )
    bad_engine = GenericTrainingEngine(bad_config)
except Exception as e:
    print(type(e).__name__ + ':', e)


ValueError: model_type must be 'sklearn' or 'pytorch'


### Error 2: Unsupported sklearn model name

This happens when the model name is not in the sklearn registry.


In [19]:
try:
    bad_config = TrainingConfig(
        model_type='sklearn',
        model_name='svm'
    )
    bad_engine = GenericTrainingEngine(bad_config)
except Exception as e:
    print(type(e).__name__ + ':', e)


ValueError: Unsupported sklearn model: svm


### Error 3: Unsupported PyTorch model name

This happens when the model name is not in the PyTorch registry.


In [21]:
try:
    bad_config = TrainingConfig(
        model_type='pytorch',
        model_name='cnn'
    )
    bad_engine = GenericTrainingEngine(bad_config)
except Exception as e:
    print(type(e).__name__ + ':', e)


ValueError: Unsupported pytorch model: cnn


### Error 4: Missing required PyTorch model parameters

For example, `SimpleMLP` needs `input_dim`. If it is missing, model creation fails.


In [23]:
try:
    pytorch_config = TrainingConfig(
        model_type='pytorch',
        model_name='simple_mlp',
        model_params={}
    )
    pytorch_engine = GenericTrainingEngine(pytorch_config)
except Exception as e:
    print(type(e).__name__ + ':', e)


TypeError: SimpleMLP.__init__() missing 1 required positional argument: 'input_dim'
